# MeetMind — WhisperX Transcription Server

Chạy từng cell theo thứ tự. Sau **Cell 4** sẽ có URL để paste vào Railway.

> **Yêu cầu:** Đổi Runtime → T4 GPU trước khi chạy  
> Runtime → Change runtime type → T4 GPU

In [ ]:
# Cell 1 — Cài đặt dependencies
# Pin numpy/scipy/sklearn trước để tránh conflict với whisperx internals
!pip install -q numpy==2.0.2 scipy==1.13.1 scikit-learn==1.5.2 --force-reinstall
!pip install -q whisperx fastapi uvicorn pyngrok nest-asyncio httpx
print("Done installing")

In [ ]:
# Cell 2 — Load WhisperX model (~2 phút lần đầu)
import torch, whisperx

device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "int8"

print(f"Device: {device}")
model = whisperx.load_model("large-v2", device, compute_type=compute_type)
print("Model loaded!")

In [ ]:
# Cell 3 — Định nghĩa FastAPI /transcribe endpoint
import nest_asyncio, tempfile, os, httpx
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

nest_asyncio.apply()
app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])

class TranscribeRequest(BaseModel):
    audio_url: str

@app.get('/health')
def health():
    return {'status': 'ok'}

@app.post('/transcribe')
async def transcribe(req: TranscribeRequest):
    async with httpx.AsyncClient(timeout=120) as client:
        r = await client.get(req.audio_url)
    if r.status_code != 200:
        raise HTTPException(status_code=400, detail='Cannot download audio')

    suffix = '.m4a' if '.m4a' in req.audio_url else '.mp3'
    with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as f:
        f.write(r.content)
        tmp = f.name

    try:
        audio = whisperx.load_audio(tmp)
        result = model.transcribe(audio, batch_size=16)
    finally:
        os.unlink(tmp)

    segments = [
        {'start': round(s['start'], 2), 'end': round(s['end'], 2), 'text': s['text'].strip()}
        for s in result['segments']
    ]
    return {'segments': segments, 'language': result.get('language', '')}

print("Server defined!")

In [ ]:
# Cell 4 — Expose qua ngrok + chạy server
# Lấy token miễn phí: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = ""  # <-- paste token vào đây

from pyngrok import ngrok
import uvicorn, threading, time, httpx

if not NGROK_TOKEN:
    raise ValueError("Paste NGROK_TOKEN vào đây!")

ngrok.set_auth_token(NGROK_TOKEN)
ngrok.kill()  # kill tunnel cũ nếu có

# Start server trước
def run():
    uvicorn.run(app, host="0.0.0.0", port=8001, log_level="warning")

t = threading.Thread(target=run, daemon=True)
t.start()

# Chờ uvicorn ready
for _ in range(20):
    try:
        httpx.get("http://localhost:8001/health", timeout=1)
        break
    except Exception:
        time.sleep(0.5)

# Sau đó mới tạo tunnel
tunnel = ngrok.connect(8001)
public_url = tunnel.public_url

print('=' * 60)
print(f"COLAB_WHISPER_URL = {public_url}/transcribe")
print('=' * 60)
print("Copy URL trên → paste vào Railway Variables → Save")

In [ ]:
# Cell 5 (tuỳ chọn) — Test health check
import httpx
r = httpx.get(f"{public_url}/health")
print(r.json())  # {'status': 'ok'}